In [14]:
import os
import math
import json
from datetime import datetime, timezone
from uuid import uuid4
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from pydantic import BaseModel, Field
from typing import Literal


class PersonState(BaseModel):
    mood: str
    energy: float = Field(ge=0, le=1)
    activity: str
    attention_focus: str
    environment: str
    current_intent: str


class Appraisal(BaseModel):
    observation_id: str = Field(description="原样返回本次观察的编号")
    attention: float = Field(ge=0, le=1, description="对这段内容的留意程度")
    emotional_impact: float = Field(
        ge=0, le=1, description="这段内容引起的情绪反应强度"
    )
    importance: float = Field(ge=0, le=1, description="这件事对角色的重要程度")
    memory_content: str | None = Field(
        description="值得长期保留的记忆正文。如果没有值得留下的内容，返回 null"
    )


class Memory(BaseModel):
    memory_id: str
    owner_id: str
    source_observation_id: str
    content: str
    created_at: datetime
    # 形成时的强度，不因后续回忆而修改。
    initial_strength: float = Field(ge=0, le=1)
    # 上次回忆后的基准强度。
    # 未回忆过时，使用 initial_strength。
    strength_after_recall: float | None = Field(
        default=None, ge=0, le=1
    )
    # 衰减时间尺度，单位：天。
    # 未设置时，根据初始强度计算。
    stability_days: float | None = Field(
        default=None, gt=0
    )
    recall_count: int = Field(default=0, ge=0)
    last_recalled_at: datetime | None = None
    # forgotten 在这个 demo 中表示很难想起，正文仍然保留。
    status: Literal["active", "fading", "forgotten"] = "active"

    def get_current_strength(
            self,
            now: datetime | None = None,
    ) -> float:
        """计算当前强度，不强化记忆。"""
        if now is None:
            now = datetime.now(timezone.utc)

        base_strength = (
            self.strength_after_recall
            if self.strength_after_recall is not None
            else self.initial_strength
        )

        # 实验映射：初始强度 0～1 对应 1～90 天。
        stability = (
            self.stability_days
            if self.stability_days is not None
            else 1 + 89 * self.initial_strength
        )

        base_time = self.last_recalled_at or self.created_at

        days_since = max(
            0.0,
            (now - base_time).total_seconds() / 86400,
        )

        strength = base_strength * math.exp(
            -days_since / stability
        )

        if strength > 0.6:
            self.status = "active"
        elif strength > 0.2:
            self.status = "fading"
        else:
            self.status = "forgotten"

        return strength

    def reinforce(
            self,
            now: datetime | None = None,
    ) -> float:
        """实际成功回忆后，恢复并强化记忆。"""
        if now is None:
            now = datetime.now(timezone.utc)

        base_time = self.last_recalled_at or self.created_at

        if now < base_time:
            raise ValueError("回忆时间不能早于当前基准时间")

        days_since = (
                             now - base_time
                     ).total_seconds() / 86400

        base_strength = (
            self.strength_after_recall
            if self.strength_after_recall is not None
            else self.initial_strength
        )

        stability = (
            self.stability_days
            if self.stability_days is not None
            else 1 + 89 * self.initial_strength
        )

        # 间隔相对于稳定性越长，这个值越接近 1。
        # 同一时刻重复调用时为 0，不会持续刷高强度。
        spacing = 1 - math.exp(
            -days_since / stability
        )

        # 延长今后的保持时间，不限制在 0～1。
        self.stability_days = stability * (1 + spacing)

        # 成功回忆后恢复到先前基准，再做少量提升。
        # 越接近 1，可以提升的空间越小。
        self.strength_after_recall = (
                base_strength
                + 0.2 * spacing * (1 - base_strength)
        )

        self.last_recalled_at = now
        self.recall_count += 1

        # 按实际强度更新状态，不强制设成 active。
        return self.get_current_strength(now)


def cosine(a: list[float], b: list[float]) -> float:
    dot = sum(x * y for x, y in zip(a, b, strict=True))
    norm = math.sqrt(sum(x * x for x in a) * sum(y * y for y in b))
    return dot / norm if norm else 0.0


def main() -> None:
    load_dotenv()

    # 该ai的唯一id
    owner_id = "user_1234567890"
    states = {
        owner_id: PersonState(
            mood="有点烦躁",
            energy=0.35,
            activity="赶作业",
            attention_focus="电脑上的作业",
            environment="手机在手边，静音，锁屏可以显示消息预览",
            current_intent="先交作业",
        )
    }

    # 获取该ai的状态
    state = states[owner_id]

    user_message_1 = (
        "明天下午两点要不要一起去图书馆？有空的话，我们在东门碰面。"
        "记得带上我上周借给你的《活着》，我想拿回去了。"
        "书里夹着一张蓝色便签，你先别扔，我还要用。"
        "我想去三楼靠窗的位置，没位置就去二楼。"
    )

    user_message_2 = (
        "周六下午三点要不要一起去城南新开的书店？我们可以在门口碰面。"
        "顺便带上我前几天借给你的《小王子》，我想拿回来给妹妹看。"
        "书里夹着一张电影票，是我留作纪念的，记得一起带回来。"
        "要是你周六没空，我们就改到周日上午。"
    )

    user_message_3 = (
        "下周一晚上六点有空来帮我搬一下书柜吗？工具我已经准备好了。"
        "来的时候，顺便带上我上个月借给你的《平凡的世界》，我想重新读一遍。"
        "那套书一共三册，你看看有没有都放在一起。"
        "搬完书柜，我请你去路口那家面馆吃饭。"
    )

    user_messages = [user_message_1, user_message_2, user_message_3]

    prompt = ChatPromptTemplate.from_messages([
        (
            "system",
            "你负责模拟角色读到几条用户消息后，会留下怎样的长期记忆。\n\n"
            "本次观察的编号是 {observation_id}，请将它原样填入 observation_id。\n"
            "角色当时的状态：\n{state}\n\n"
            "用户消息是角色实际看到的内容。结合角色的精力、当前活动和注意力焦点，"
            "判断他对这条消息有多留意，并据此选择要记住的细节。"
            "分心时优先保留主线、对后续行动有用的细节，以及让角色印象深刻的内容。"
            "专注时可以保留更多有辨识度的细节。只扫到或记不清的次要细节可以省略。\n\n"
            "从角色视角写记忆。用户原话中的“我”指用户，“你”指角色。"
            "改写人称时，保留谁对谁做了什么，尤其注意借出、借入、归还等动作的方向。"
            "保留会改变原意的条件和不确定性。用户提出的邀请仍是邀请，"
            "只有消息明确表明双方已经确认，才写成已确定的约定。"
            "拿不准的细节可以省略，避免猜测或补充消息里没有的信息。\n\n"
            "分别给以下三个维度打分，每项为 0 到 1：\n"
            "- attention：角色当时对这条消息的实际留意程度。\n"
            "- emotional_impact：这条消息本身给角色带来的情绪反应强度。"
            "角色原有的心情只作为判断背景。\n"
            "- importance：这件事对角色的目标、关系或后续行动有多重要。"
            "即使当时分心或情绪波动很小，事情本身也可能很重要。\n\n"
            "如果这次没有值得长期保留的内容，memory_content 返回 null。",
        ),
        ("human", "{user_message}"),
    ])

    model = ChatOpenAI(
        model=os.environ["LLM_MODEL"],
        base_url=os.getenv("LLM_BASE_URL"),
        api_key=os.environ["LLM_API_KEY"],
        temperature=0.7,
    )

    chain = prompt | model.with_structured_output(
        Appraisal,
        method="function_calling",
    )

    memories: dict[str, Memory] = {}

    for user_message in user_messages:
        observation_id = f"obs_{uuid4()}"

        appraisal = chain.invoke({
            "observation_id": observation_id,
            "state": state.model_dump_json(),
            "user_message": user_message,
        })

        if appraisal is None:
            raise RuntimeError("模型没有返回结构化的 Appraisal")

        if appraisal.observation_id != observation_id:
            raise ValueError("模型返回的 observation_id 与本次观察不一致")

        # print(appraisal.model_dump_json(indent=2))

        content = (appraisal.memory_content or "").strip()
        if not content:
            print("这次观察没有形成长期记忆。")
            return

        initial_strength = round(
            0.3 * appraisal.attention
            + 0.3 * appraisal.emotional_impact
            + 0.4 * appraisal.importance,
            3,
        )

        memory_id = str(uuid4())

        memories[memory_id] = Memory(
            memory_id=memory_id,
            owner_id=owner_id,
            source_observation_id=appraisal.observation_id,
            content=content,
            created_at=datetime.now(timezone.utc),
            initial_strength=initial_strength,
        )

    print(json.dumps(
        {k: v.model_dump() for k, v in memories.items()},
        ensure_ascii=False,
        indent=2,
        default=str,
    ))
    print("======================")

    embeddings = OpenAIEmbeddings(
        model=os.environ["EMBEDDING_MODEL"],
        base_url=os.getenv("EMBEDDING_BASE_URL"),
        api_key=os.environ["EMBEDDING_API_KEY"],
        check_embedding_ctx_length=False,
        request_timeout=30,
        max_retries=0,
    )

    # 先限定持有人。索引结构：memory_id -> (正文版本, 向量)。
    records = [m for m in memories.values() if m.owner_id == owner_id]

    vectors = embeddings.embed_documents([m.content for m in records])

    index = {
        m.memory_id: vector
        for m, vector in zip(records, vectors, strict=True)
    }

    query = "我们什么时候去学校"
    query_vector = embeddings.embed_query(query)

    # 本次提取统一使用同一个时间。
    retrieval_time = datetime.now(timezone.utc)
    candidates = []

    for memory_id, vector in index.items():
        memory = memories[memory_id]

        if memory.owner_id != owner_id:
            continue

        # 查询与记忆的语义相似度。
        similarity = cosine(query_vector, vector)

        # 使用同一个提取时间，计算当前强度并刷新状态。
        strength = memory.get_current_strength(retrieval_time)

        # 相似度与当前强度共同参与排序。
        score = 0.8 * similarity + 0.2 * strength

        candidates.append(
            (memory_id, similarity, strength, score)
        )

    candidates.sort(key=lambda item: item[3], reverse=True)

    print("回忆线索：", query)
    for memory_id, similarity, strength, score in candidates[:2]:
        memory = memories[memory_id]

        print(
            f"记忆 ID：{memory_id}\n"
            f"相似度：{similarity:.4f}\n"
            f"初始强度：{memory.initial_strength:.4f}\n"
            f"当前强度：{strength:.4f}\n"
            f"综合分：{score:.4f}\n"
            f"正文：{memory.content}\n"
        )


if __name__ == "__main__":
    main()

{
  "0c9f89d1-88d8-4cf8-b7d5-4bc60c4c62c1": {
    "memory_id": "0c9f89d1-88d8-4cf8-b7d5-4bc60c4c62c1",
    "owner_id": "user_1234567890",
    "source_observation_id": "obs_0cc49eff-3f20-4369-b861-50aff9cc177e",
    "content": "我正赶作业，瞄到用户的消息：他问明天下午两点要不要一起去图书馆，有空的话在东门碰面——只是问，我还没答应。他还让我把上周借我的《活着》带回去，特意叮嘱书里夹的那张蓝色便签先别扔，他还要用。另外他说想去三楼靠窗的位置，没位置就去二楼。借书要还、便签不能扔，这两件事我记住了。",
    "created_at": "2026-11-05 08:55:59.914694+00:00",
    "initial_strength": 0.535,
    "strength_after_recall": null,
    "stability_days": null,
    "recall_count": 0,
    "last_recalled_at": null,
    "status": "active"
  },
  "1e872233-77af-425b-882d-cf76f8d414ca": {
    "memory_id": "1e872233-77af-425b-882d-cf76f8d414ca",
    "owner_id": "user_1234567890",
    "source_observation_id": "obs_161a3667-71bf-4800-b254-5e0a6a11a83f",
    "content": "用户邀请我周六下午三点去城南新开的书店，在门口碰面；如果我周六没空，就改到周日上午。用户还让我带上前几天借给我的《小王子》，他/她想拿回去给妹妹看，并提醒书里夹着一张他/她留作纪念的电影票，要一起带回来。",
    "created_at": "2026-11-05 08:56:13.647356+00:00",
    "initial_strength